In [3]:
import pandas as pd
import numpy as np

# Cargar dataset
df = pd.read_csv('./training.csv')

# 1. Limpieza de strings (eliminar tabulaciones y saltos de línea)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

# 2. Tratamiento de nulos
# Imputamos moda para categóricos y mediana para numéricos
df['Gender'].fillna(df['Gender'].mode()[0], inplace=True)
df['Workout_Type'].fillna('Unknown', inplace=True)
df['Experience_Level'].fillna(df['Experience_Level'].median(), inplace=True)

# 3. Recálculo del BMI (Corrección de error de lógica)
df['BMI'] = df['Weight (kg)'] / (df['Height (m)'] ** 2)

# 4. Corrección de tipos de datos
cols_to_int = ['Age', 'Workout_Frequency (days/week)', 'Experience_Level']
for col in cols_to_int:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# 5. Limpieza específica de columnas con basura
df['Max_BPM'] = pd.to_numeric(df['Max_BPM'], errors='coerce')

# 6. Normalización estructural: Convertir espacios vacíos, tabs o celdas invisibles en NaN
# Sin este paso, fillna() no detectará los huecos que mencionaste antes
df = df.replace(r'^\s*$', np.nan, regex=True)

# 7. Tratamiento para Columnas Numéricas (Media)
# Edad, Peso, Calorías, BPMs, etc.
columnas_numericas = df.select_dtypes(include=[np.number]).columns
for col in columnas_numericas:
    if df[col].isnull().any():
        media = df[col].mean()
        df[col] = df[col].fillna(media)

# 8. Tratamiento para Columnas Categóricas (Moda)
# Gender y Workout_Type (que tenían blancos en las filas 46, 58, 12, etc.)
columnas_categoricas = df.select_dtypes(include=['object']).columns
for col in columnas_categoricas:
    if df[col].isnull().any():
        # Calculamos la moda: .mode() devuelve una serie, tomamos el índice [0]
        moda = df[col].mode()[0]
        df[col] = df[col].fillna(moda)
        print(f"Columna '{col}': Rellenada con la moda -> '{moda}'")

# 9. Corrección de tipado Post-Imputación
# Al imputar medias, algunas columnas que deberían ser enteras pueden quedar como float
# Las convertimos para que en los gráficos se vean limpias (ej. Edad 25 en vez de 25.0)
cols_a_entero = ['Age', 'Workout_Frequency (days/week)', 'Experience_Level']
for col in cols_a_entero:
    df[col] = df[col].astype(int)

# 10. Verificación de éxito
print("-" * 30)
print("Limpieza Completa. Valores nulos encontrados:")
print(df.isnull().sum())
print("-" * 30)
print("Dataset listo para visualización de alto impacto.")


------------------------------
Limpieza Completa. Valores nulos encontrados:
Age                              0
Gender                           0
Weight (kg)                      0
Height (m)                       0
Max_BPM                          0
Avg_BPM                          0
Resting_BPM                      0
Session_Duration (hours)         0
Calories_Burned                  0
Workout_Type                     0
Fat_Percentage                   0
Water_Intake (liters)            0
Workout_Frequency (days/week)    0
Experience_Level                 0
BMI                              0
dtype: int64
------------------------------
Dataset listo para visualización de alto impacto.


In [5]:
df.sample(50)

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
1479,20,Male,40.000000,1.78,192.0,149.000000,57.000000,0.930000,1075.0,Strength,25.500000,1.9,4,1,12.624669
45,37,Female,55.400000,1.68,197.0,146.259322,52.000000,1.040000,338.0,Cardio,27.400000,3.2,0,3,19.628685
1791,32,Male,40.600000,1.92,166.0,169.000000,57.000000,1.330000,1783.0,Strength,17.100000,3.6,5,3,11.013455
1073,36,Male,78.200000,1.63,191.0,169.000000,74.000000,2.000000,1381.0,Cardio,23.100000,2.7,3,2,29.432798
1005,22,Male,129.900000,1.62,185.0,141.000000,52.000000,1.620000,571.0,Strength,27.200000,1.8,4,2,49.497028
400,20,Male,87.100000,1.91,186.0,150.000000,51.000000,1.390000,1431.0,Yoga,27.000000,2.8,3,3,23.875442
395,31,Male,58.700000,1.66,183.0,149.000000,74.000000,1.150000,1368.0,Cardio,23.100000,2.3,4,1,21.302076
1517,34,Female,79.100000,1.73,188.0,151.000000,58.000000,1.320000,1191.0,Yoga,30.700000,3.6,0,2,26.429216
278,18,Female,50.100000,1.80,174.0,125.000000,50.000000,1.380000,1185.0,HIIT,30.800000,3.4,3,2,15.462963
419,38,Female,40.000000,1.88,164.0,154.000000,70.000000,1.630000,1109.0,Yoga,20.700000,1.7,2,2,11.317338
